# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values. This will help you identify how the data is structured and which record sets or fields to explore further.

We will display all available record sets and their fields, referencing each entity by its `@id`.

In [ ]:
# List all record sets with their @id
print('Available Record Sets:')
for record_set in metadata.record_sets:
    print(f"- @id: {record_set['@id']}, Name: {record_set.get('name', 'N/A')}")

# Display fields for each record set
print('\nFields by Record Set:')
for record_set in metadata.record_sets:
    print(f"\nRecord Set @id: {record_set['@id']}, Name: {record_set.get('name', 'N/A')}")
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        # Each field is represented by its @id and optionally a name
        print(f"    Field @id: {field['@id']}, Name: {field.get('name', 'N/A')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview to access the desired data for downstream analysis.

In [ ]:
# List all record_set @id values
record_sets_ids = [r['@id'] for r in metadata.record_sets]
print('Record Set IDs:', record_sets_ids)

# Load each record set as a DataFrame
dataframes = {}
for record_set_id in record_sets_ids:
    print(f'Loading records from Record Set @id: {record_set_id}')
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    # Preview the columns
    print(f"Columns in '{record_set_id}':", df.columns.tolist())
    print(df.head(), '\n')
# Choose a record set to work with (first one for this example):
main_record_set_id = record_sets_ids[0] if record_sets_ids else None
if main_record_set_id:
    print('Previewing main record set:', main_record_set_id)
    print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates how to identify numeric fields and perform simple analyses, always referencing columns by their `@id` (from above).

*Note*: Update the `numeric_field_id` and `group_field_id` variables with the appropriate field `@id` values for your analysis.

In [ ]:
# Identify numeric fields in the chosen record set DataFrame
df = dataframes.get(main_record_set_id)
if df is not None and not df.empty:
    numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
    print('Numeric fields:', numeric_fields)
    if numeric_fields:
        # Choose a numeric field by its @id
        numeric_field_id = numeric_fields[0]  # You can change this to another field
        print(f"Using numeric field: {numeric_field_id}")
        threshold = df[numeric_field_id].mean()  # Use the mean as an example filter
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the selected numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a non-numeric field (if available)
        candidate_group_fields = [c for c in df.columns if c != numeric_field_id and not pd.api.types.is_numeric_dtype(df[c])]
        print('Candidate group fields:', candidate_group_fields)
        if candidate_group_fields:
            group_field_id = candidate_group_fields[0]  # By default, select the first string/categorical field
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
        else:
            group_field_id = None
    else:
        print('No numeric fields available for EDA.')
else:
    print('No records found for main record set.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. You can use matplotlib or seaborn, always referencing columns by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and not df.empty and numeric_fields:
    # Histogram of the selected numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id is not None:
        # Boxplot grouped by a categorical field
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, you learned how to load and explore a dataset described by a Croissant schema using `mlcroissant`. 

Key steps included:
- Reviewing available record sets and fields using their `@id`
- Loading data dynamically
- Conducting basic EDA and visualizations referencing all fields by their `@id`

Continue exploration with more advanced analyses and by consulting the full Croissant schema for additional context and attributes!